# Modeling & Evaluation — Família 06: Boosting de árvores

**Projeto:** Previsão do vencedor de confrontos profissionais de CS2
**Dataset:** [Counter-Strike Pro Matches (Kaggle, ektarr)](https://www.kaggle.com/datasets/ektarr/counter-strike-pro-matches)

| | |
|---|---|
| **Família de modelos** | **Boosting de árvores** |
| **Modelo(s) neste notebook** | Gradient Boosting, HistGradientBoosting, XGBoost, LightGBM |
| **Alvo** | `target` = 1 se a equipa 1 vence o jogo/mapa, 0 caso contrário |
| **Entrada** | `cs2_matches_train.csv` e `cs2_matches_test.csv` em `MyDrive/CS2_Projeto/` (gerados no Data Preparation, split cronológico 80/20, sem *leakage*) |
| **Features** | 12 pré-jogo: Elo (`elo_team1`, `elo_team2`, `elo_diff`), taxa de vitórias, forma recente e h2h |
| **Validação** | `TimeSeriesSplit` (5 folds) com `GridSearchCV`, métrica de seleção: accuracy |

O Elo é usado como feature de treino, tal como as restantes. Não há baseline Elo e, nas métricas, aparece apenas o desempenho dos modelos.

## Sobre esta família

O ***boosting*** constrói árvores pequenas **em sequência**, cada uma a corrigir os erros das anteriores. Costuma ser a família mais forte em dados tabulares.

- **Gradient Boosting** (scikit-learn): a versão clássica.
- **HistGradientBoosting** (scikit-learn): usa histogramas, muito mais rápido.
- **XGBoost** e **LightGBM**: implementações otimizadas; entram automaticamente se estiverem instaladas (no Colab já vêm instaladas).

**Limitações:** muitos hiperparâmetros e tendência a overfitting se for demasiado profundo ou tiver demasiadas árvores; por isso se usam taxas de aprendizagem baixas e árvores pouco profundas.

## 1. Setup

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
RANDOM_STATE = 42

In [ ]:
FAMILIA = 'Boosting de árvores'

## 2. Carregamento dos dados

No Colab, monta o Google Drive e lê os CSV de `MyDrive/CS2_Projeto/` (a mesma pasta onde o Data Preparation os guarda). Se não os encontrar lá, tenta a pasta atual da sessão. Fora do Colab, usa a pasta atual.

In [ ]:
# Pasta no Google Drive onde o Data Preparation guardou os CSV (tem de ser a mesma usada lá)
PASTA_DRIVE = '/content/drive/MyDrive/CS2_Projeto/'
FICHEIROS = ['cs2_matches_train.csv', 'cs2_matches_test.csv']

PASTA = '.'   # fora do Colab usa-se a pasta atual
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(PASTA_DRIVE, exist_ok=True)
    PASTA = PASTA_DRIVE
except ImportError:
    print("Fora do Colab: a usar a pasta atual.")

caminhos = {f: os.path.join(PASTA, f) for f in FICHEIROS}

if not all(os.path.exists(c) for c in caminhos.values()):
    if all(os.path.exists(f) for f in FICHEIROS):
        # ficheiros soltos na sessão (ex.: /content/) em vez do Drive
        caminhos = {f: f for f in FICHEIROS}
        print("Ficheiros não encontrados no Drive; a usar os da pasta atual da sessão.")
    else:
        em_falta = [c for c in caminhos.values() if not os.path.exists(c)]
        raise FileNotFoundError(
            f"Não encontrei {em_falta}. Corre a secção 10 do Data Preparation a guardar em {PASTA_DRIVE}")

train = pd.read_csv(caminhos['cs2_matches_train.csv'])
test = pd.read_csv(caminhos['cs2_matches_test.csv'])
print("Dados lidos de:", os.path.dirname(os.path.abspath(caminhos['cs2_matches_train.csv'])))

X_train, y_train = train.drop(columns=['target']), train['target']
X_test, y_test = test.drop(columns=['target']), test['target']

assert list(X_train.columns) == list(X_test.columns), "Treino e teste com colunas diferentes"
assert not X_train.isna().any().any() and not X_test.isna().any().any(), "Existem valores em falta"

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 3. Modelo(s) e grelha de hiperparâmetros

In [ ]:
def com_escala(modelo):
    return Pipeline([('scaler', StandardScaler()), ('clf', modelo)])

In [ ]:
modelos = {
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [2, 3]}),

    'HistGradientBoosting': (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {'learning_rate': [0.03, 0.1], 'max_depth': [3, 6], 'min_samples_leaf': [20, 50]}),
}

try:
    from xgboost import XGBClassifier
    modelos['XGBoost'] = (
        XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
        {'n_estimators': [100, 300], 'learning_rate': [0.03, 0.1], 'max_depth': [2, 4]})
except ImportError:
    print("XGBoost não está instalado — ignorado.")

try:
    from lightgbm import LGBMClassifier
    modelos['LightGBM'] = (
        LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
        {'n_estimators': [100, 300], 'learning_rate': [0.03, 0.1], 'num_leaves': [7, 15]})
except ImportError:
    print("LightGBM não está instalado — ignorado.")

## 4. Treino com validação cruzada temporal

Cada fold treina nos jogos mais antigos e valida nos seguintes, para a seleção de hiperparâmetros respeitar a ordem cronológica.

In [ ]:
cv_temporal = TimeSeriesSplit(n_splits=5)

melhores = {}
registo_cv = []

for nome, (estimador, grelha) in modelos.items():
    t0 = time.time()
    gs = GridSearchCV(estimador, grelha, cv=cv_temporal, scoring='accuracy', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    melhores[nome] = gs.best_estimator_
    registo_cv.append({
        'Família': FAMILIA,
        'Modelo': nome,
        'Accuracy CV (média)': gs.best_score_,
        'Melhores hiperparâmetros': gs.best_params_,
        'Tempo (s)': round(time.time() - t0, 1),
    })
    print(f"{nome:<22} CV accuracy = {gs.best_score_:.4f}  ({time.time() - t0:.1f}s)")

df_cv = pd.DataFrame(registo_cv).sort_values('Accuracy CV (média)', ascending=False).reset_index(drop=True)
df_cv

## 5. Avaliação no conjunto de teste

Jogos mais recentes, nunca vistos no treino. Métricas: accuracy, precision, recall, F1 e ROC AUC.

In [ ]:
def pontuacao(modelo, X):
    if hasattr(modelo, 'predict_proba'):
        return modelo.predict_proba(X)[:, 1]
    return modelo.decision_function(X)

linhas, previsoes = [], {}
for nome, modelo in melhores.items():
    y_pred = modelo.predict(X_test)
    previsoes[nome] = y_pred
    linhas.append({
        'Família': FAMILIA,
        'Modelo': nome,
        'Accuracy (treino)': accuracy_score(y_train, modelo.predict(X_train)),
        'Accuracy (teste)': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC AUC': roc_auc_score(y_test, pontuacao(modelo, X_test)),
    })

resultados = (pd.DataFrame(linhas)
              .sort_values('Accuracy (teste)', ascending=False)
              .reset_index(drop=True))
resultados['Gap treino-teste'] = resultados['Accuracy (treino)'] - resultados['Accuracy (teste)']
resultados.round(4)

### 5.1 Comparação entre os modelos da família

In [ ]:
ordem = resultados.sort_values('Accuracy (teste)')
fig, ax = plt.subplots(figsize=(8, 1.2 + 0.7 * len(ordem)))
barras = ax.barh(ordem['Modelo'], ordem['Accuracy (teste)'], color='#4C78A8')
ax.bar_label(barras, fmt='%.3f', padding=3)
ax.set_xlim(max(0, ordem['Accuracy (teste)'].min() - 0.05), min(1, ordem['Accuracy (teste)'].max() + 0.05))
ax.set_xlabel('Accuracy no conjunto de teste')
ax.set_title(f'{FAMILIA} — accuracy de teste por modelo')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 Treino vs teste

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(resultados))
ax.bar(x - 0.2, resultados['Accuracy (treino)'], width=0.4, label='Treino', color='#9ecae9')
ax.bar(x + 0.2, resultados['Accuracy (teste)'], width=0.4, label='Teste', color='#4C78A8')
ax.set_xticks(x)
ax.set_xticklabels(resultados['Modelo'], rotation=20, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Treino vs teste (diferença grande = overfitting)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Melhor modelo da família

Matriz de confusão e importância das features (por permutação, no conjunto de teste).

In [ ]:
nome_melhor = resultados.loc[0, 'Modelo']
melhor = melhores[nome_melhor]
print(f"Família: {FAMILIA}")
print(f"Melhor modelo no teste: {nome_melhor}")
print(f"Accuracy de teste: {resultados.loc[0, 'Accuracy (teste)']:.4f}")
print(f"Hiperparâmetros: {df_cv.set_index('Modelo').loc[nome_melhor, 'Melhores hiperparâmetros']}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, previsoes[nome_melhor]),
                       display_labels=['Equipa 2 vence', 'Equipa 1 vence']).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Matriz de confusão — {nome_melhor}')

perm = permutation_importance(melhor, X_test, y_test, scoring='accuracy',
                              n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1)
imp = (pd.DataFrame({'Feature': X_test.columns,
                     'Queda de accuracy': perm.importances_mean,
                     'Desvio-padrão': perm.importances_std})
       .sort_values('Queda de accuracy', ascending=True))
axes[1].barh(imp['Feature'], imp['Queda de accuracy'], xerr=imp['Desvio-padrão'], color='#F58518')
axes[1].set_xlabel('Queda média de accuracy ao baralhar a feature')
axes[1].set_title('Importância das features (permutação, teste)')
axes[1].grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Guardar resultados

Os resultados e o modelo ficam em `CS2_Projeto/resultados/` (no Drive, se estiver montado), por isso não se perdem quando a sessão termina.

In [ ]:
PASTA_SAIDA = os.path.join(PASTA, 'resultados')
os.makedirs(PASTA_SAIDA, exist_ok=True)

resultados.round(4).to_csv(os.path.join(PASTA_SAIDA, 'resultados_boosting.csv'), index=False)
df_cv.to_csv(os.path.join(PASTA_SAIDA, 'resultados_cv_boosting.csv'), index=False)
joblib.dump(melhor, os.path.join(PASTA_SAIDA, 'modelo_boosting.joblib'))
print("Ficheiros guardados em:", os.path.abspath(PASTA_SAIDA))
print("  resultados_boosting.csv, resultados_cv_boosting.csv, modelo_boosting.joblib")

## 8. Notas para o relatório

- Família **Boosting de árvores**: usar este notebook para a secção da família e comparar os `resultados_boosting.csv` das 7 famílias, todos na pasta `CS2_Projeto/resultados/`.
- Com ~2 mil jogos de teste, a margem de erro da accuracy ronda os ±2 pontos percentuais; diferenças pequenas entre famílias podem ser ruído.
- A coluna `Gap treino-teste` indica overfitting quando é grande e positiva.
- O modelo prevê o vencedor de cada **mapa**; para o confronto completo (BO3/BO5) seria preciso agregar por `match_id`.